# 02 Storm detection and validation
Rules: see `reports/METHODOLOGY.md` section 3.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np, json
import matplotlib.pyplot as plt
from src.common import load_config, resolve
cfg = load_config(ROOT / 'config.yaml')

In [ ]:
from src.storm_detection import Params, detect_storms
from src.catalogue_validation import validate_catalogue
clean = pd.read_csv(resolve(cfg,'clean_series'), parse_dates=['timestamp_utc']).set_index('timestamp_utc')
p = Params(**{**Params.from_config(cfg).__dict__, 'tiers': cfg['detection']['tiers']})
cat = detect_storms(clean.dst_nt, p, clean['product'])
print(len(cat), 'storms'); print('validation problems:', validate_catalogue(cat, clean.dst_nt, p))

In [ ]:
cat.to_csv(resolve(cfg,'catalogue'), index=False)   # or: python -m src.storm_detection
cat.head()

In [ ]:
# parameter sensitivity: how much do the choices matter?
import itertools
rows=[]
for rel,gap,minc in itertools.product([-15,-20,-25],[3,6,12],[1,2]):
    q=Params(**{**p.__dict__,'release_nt':rel,'merge_gap_hours':gap,'min_core_hours':minc})
    c=detect_storms(clean.dst_nt,q); rows.append((rel,gap,minc,len(c),c.total_duration_hours.median(),c.total_duration_hours.max()))
pd.DataFrame(rows,columns=['release','merge_gap','min_core','n','median_h','max_h'])